# Lektion 18: Säkerställa AI-agenter med kryptografiska kvitton

## Praktisk notebook

Den här notebooken går igenom fyra övningar:

1. **Signera ditt första kvitto** för ett agentverktyg och verifiera det.
2. **Manipulera kvittot** och se verifikationen misslyckas.
3. **Bygg en kedja med tre kvitton** och bekräfta kedjans integritet.
4. **Paketer en Microsoft Agent Framework-verktygssamtal** så att varje handling genererar ett kvitto.

Alla kryptografiska primitiv hämtas från väl underhållna bibliotek (`pynacl` för Ed25519, `jcs` för RFC 8785 kanoniskt JSON, `hashlib` från Pythons standardbibliotek för SHA-256). Själva kvittologiken är ren Python som du kan läsa och ändra.

Kör cellerna i ordning. Varje avsnitt är kort och fristående.


## Installera

Installera de två beroenden. Båda har tillåtande licenser (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Hjälpverktyg

Dessa två hjälpverktyg hanterar base64url-kodning (utan utfyllnad) och SHA-256-hashning av godtyckliga objekt. De håller resten av anteckningsboken fokuserad på kvittologiken i sig.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: Underteckna ditt första kvitto

Föreställ dig att vår agent för **Contoso Travel** just har sökt flyg från Sydney till Los Angeles för en kund. Vi vill registrera detta verktygsanrop som ett undertecknat kvitto så att en framtida revisor kan verifiera det utan att behöva lita på oss.

### Steg 1.1: Generera en signeringsnyckel

I produktion skulle agentens signeringsnyckel finnas i en hårdvarusäkerhetsmodul (HSM), Azure Key Vault eller en liknande skyddad lagring. För denna lektion genererar vi en ny nyckel i minnet.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Steg 1.2: Bygg kvittots nyttolast

Nyttolasten innehåller allt vi vill att kvittot ska styrka: vem som agerade, vilket verktyg, med vilka argument, vad som kom tillbaka, under vilken policy och när. Vi hashar argumenten och resultatet istället för att inkludera dem inline så att kvittot inte läcker känsligt innehåll.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Steg 1.3: Signera och sammanställ kvittot

Tre steg:

1. Kanonikalsera nyttolasten med JCS så att två implementationer som producerar samma logiska kvitto genererar byte-identiska bytes.
2. Hasha de kanonikaliserade bytesen med SHA-256.
3. Signera hashen med Ed25519-privatnyckeln.

Signaturen bifogas sedan den ursprungliga nyttolasten för att producera det slutliga kvittot.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Steg 1.4: Verifiera kvittot

Verifieringen vänder processen. Vi tar bort signaturen, beräknar den kanoniska hashen på nytt och kontrollerar signaturen mot den publika nyckeln i kvittot.

En granskare som utför denna verifiering behöver ingenting från oss förutom själva kvittot. Ingen tjänst att anropa, ingen nyckelkatalog att fråga, inget förtroende krävs.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Du bör se `Receipt is valid: True`. Agenten har producerat sin första kryptografiskt signerade revisionspost.


## Sektion 2: Manipulera kvittot

Hela poängen med kvitton är att de är manipulationssäkra. Låt oss bevisa det.

Vi kommer att ändra exakt ett tecken i kvittot och se verifieringen misslyckas.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Vad hände just?

När vi ändrade `policy_id` ändrades de kanoniska byten. SHA-256-hashen av dessa byte ändrades. Signaturen (som var över den ursprungliga hashen) matchar inte längre den nya hashen. Verifieringen returnerar korrekt `False`.

Det finns inget sätt att ändra något fält i kvittot och fortfarande få det att verifieras, om inte angriparen har den privata nyckeln. Så länge den privata nyckeln finns i en nyckelvalv och den offentliga nyckeln är publicerad är det omöjligt att dölja manipulering.

Prova själv: ändra `tool_name` eller `agent_id` eller `timestamp` i cellen ovan och kör igen. Varje ändring ger ett ogiltigt kvitto.


## Sektion 3: Länka kvitton tillsammans

Ett enskilt kvitto skyddar en handling. De flesta agenter utför många handlingar. För att göra hela sekvensen manipulationssäker länkar vi varje kvitto till det föregående genom att inkludera det föregående kvittots hash i den nya kvittots innehåll.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Om någon tar bort eller byter plats på ett kvitto bryts kedjan exakt vid den punkten. Verifieringen av ett senare kvitto misslyckas eftersom dess `previous_receipt_hash` inte längre matchar den faktiska hashen av dess föregångare.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Bryt nu kedjan genom att manipulera kvittot i mitten och verifiera igen. Det manipulerade kvittot misslyckas med sin signaturkontroll, OCH det nästa kvittot misslyckas med sin kedjelänk-kontroll (eftersom dess `previous_receipt_hash` inte längre matchar det modifierade mitt-kvittots hash).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Kvitto 0 verifieras fortfarande (det ändrades inte och har ingen föregångare att vara beroende av). Kvitto 1 misslyckas med signaturkontrollen eftersom vi ändrade `tool_args_hash`. Kvitto 2 misslyckas med kedjelänkkontrollen eftersom dess `previous_receipt_hash` beräknades mot det ursprungliga (nu modifierade) kvitto 1.

Även om en angripare skriver under det modifierade kvitto 1 igen (vilket de inte kan göra utan privat nyckel), skulle kedjelänksfel i kvitto 2 fortfarande avslöja manipuleringen. För att dölja ändringen skulle angriparen behöva skriva under varje kvitto från ändringspunkten och framåt, vilket kräver innehav av den privata nyckeln.


## Avsnitt 4: Wrappa ett agentverktygssamtal med kvitto-signering

I en verklig distribution vill du inte att varje agentförfattare ska komma ihåg att anropa `make_receipt`. Du vill att kvittosignering ska ske automatiskt för varje verktygsanrop.

Här är det enklaste mönstret: en wrapper-klass som tar vilken anropbar verktygsfunktion som helst och returnerar en kvittoutsändande version av den. Detta anpassar sig till vilket agentramverk som helst, inklusive Microsoft Agent Framework (`agent_framework.azure`).

Om du inte har ett Azure AI Foundry-projekt uppsatt, visar den lokala mocken nedan ändå mönstret.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Integrering med Microsoft Agent Framework

Ovanstående `ReceiptedTool`-wrapper är ramverksoberoende. För att använda den inom en agent byggd med Microsoft Agent Framework, registrera den inlindade funktionen som ett verktyg. En skiss (du skulle ersätta mocken med en verklig Azure AI Foundry-verktygsregistrering):

```python
# Pseudokod som visar integrationsformen.
# från agent_framework.azure import AzureAIProjectAgentProvider
# från azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instruktioner="Du är en Contoso Travel-agent ...",
#     verktyg=[receipted_lookup],   # det omslutna verktyget, inte den råa funktionen
# )
# svar = agent.run("Hitta flyg från Sydney till Los Angeles i juni.")
#
# # Efter körningen har varje verktygsanrop som agenten gjorde ett signerat kvitto:
# audit_chain = receipted_lookup.receipts
```

Agentramverket behöver inte veta något om kvitton. Kvitto-signering är inlindad runt verktyget, inte inbyggt i ramverket. Så här lägger du till proveniens till befintlig agentkod utan att skriva om agenten.


## Sammanfattning och utmaning

Du har:

- Genererat ett Ed25519-nyckelpar.
- Skapat och signerat ett kvitto för ett agentverktygsanrop.
- Verifierat kvittot offline med endast den publika nyckeln.
- Manipulerat ett kvitto och observerat verifieringsfel.
- Byggt en hash-kedjad sekvens av tre kvitton.
- Manipulerat i mitten av kedjan och observerat både signaturfel och kedjelänksfel.
- Paketera ett verktygsfunktion med automatisk kvittosignering.

**Utmaning.** Utöka kvittoschemat med ett fält `request_id` (en UUID för distribuerad spårning). Uppdatera `make_receipt` för att inkludera detta, och bekräfta att kvitton fortfarande verifieras helt och hållet. Modifiera sedan fältet efter signering och bekräfta att verifiering misslyckas. Detta tvingar dig att internalisera hur varje byte i den kanoniska kodningen bidrar till signaturen.

**Viktig gräns.** Kvitton bevisar tre saker och endast tre saker: attribution (denna nyckel signerade detta innehåll), integritet (innehållet har inte ändrats sedan signering) och ordning (detta kvitto kom efter det kvittot). De bevisar INTE att agentens handling var korrekt, att policyn med `policy_id` faktiskt utvärderades, eller att agenten följde alla regler. Kvitton är en grund. Styrning är systemet du bygger ovanpå.

Läs lektionens README igen med denna gräns i åtanke. Det vanligaste misstaget team gör med kvitton är att anta att "vi har kvitton" betyder "vi har styrning." Det gör de inte. Kvitton gör agentbeteende auditerbart. De gör det inte korrekt.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, var vänlig notera att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål bör betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår till följd av användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
